
# ACLED Jordan Events - Incremental Ingestion Pipeline

## Overview
This notebook ingests ACLED (Armed Conflict Location & Event Data) for Jordan using **incremental loading**.

## Features
* **Incremental loading**: Only pulls new records since last sync (uses timestamp field)
* **Merge logic**: Upserts based on `event_id_cnty` to handle updates
* **Partitioned storage**: Table partitioned by year for efficient queries
* **API pagination**: Handles large datasets via paginated requests

## Setup Steps
1. Install Databricks CLI to create secret keys for ACLED login account
2. Create secret keys using: `databricks secrets create-scope acled` and `databricks secrets put-secret acled username/password`
3. Run the pipeline - first run does full load, subsequent runs pull only new data


In [0]:
import requests
import pandas as pd
from datetime import datetime

# -----------------------------
# 1. Login to ACLED
# -----------------------------

session = requests.Session()

login_url = "https://acleddata.com/user/login?_format=json"

# created secrets using databricks CLI in terminal
payload = {
    "name": dbutils.secrets.get("acled", "username"),
    "pass": dbutils.secrets.get("acled", "password")
}

login_response = session.post(login_url, json=payload)
login_response.raise_for_status()

csrf_token = login_response.json()["csrf_token"]

headers = {
    "X-CSRF-Token": csrf_token,
    "Content-Type": "application/json"
}

In [0]:
# -----------------------------
# Helper Functions: Schema & Quality Validation
# -----------------------------

def validate_schema(current_df, table_name):
    """
    Detect schema changes by comparing current DataFrame to existing table.
    Raises exception if breaking changes detected.
    """
    try:
        existing_table = spark.table(table_name)
        existing_columns = set(existing_table.columns)
        current_columns = set(current_df.columns)
        
        new_columns = current_columns - existing_columns
        removed_columns = existing_columns - current_columns
        
        if new_columns:
            print(f"⚠️  WARNING: New columns detected: {new_columns}")
            
        if removed_columns:
            print(f"🚨 ERROR: Columns removed: {removed_columns}")
            raise ValueError(f"Schema breaking change: columns removed {removed_columns}")
            
        # Check data types
        for col in current_columns.intersection(existing_columns):
            current_type = dict(current_df.dtypes)[col]
            existing_type = dict(existing_table.dtypes)[col]
            
            if current_type != existing_type:
                print(f"🚨 ERROR: Type change in column '{col}': {existing_type} → {current_type}")
                raise ValueError(f"Data type changed for column: {col}")
                
        print("✓ Schema validation passed")
        return True
        
    except Exception as e:
        if "Table or view not found" in str(e):
            print("First load - no schema to validate against")
            return True
        raise

def check_data_quality(df):
    """
    Validate data quality metrics and report issues.
    Returns dict with issues found.
    """
    from pyspark.sql.functions import col, countDistinct, to_date, current_date
    
    total_rows = df.count()
    issues = []
    
    # Check 1: Empty dataset
    if total_rows == 0:
        return {"total_rows": 0, "issues": ["CRITICAL: No records in dataset"]}
    
    # Check 2: Null rates in critical columns
    critical_columns = ["event_id_cnty", "event_date", "year", "latitude", "longitude"]
    
    for col_name in critical_columns:
        if col_name in df.columns:
            null_count = df.filter(col(col_name).isNull()).count()
            null_rate = (null_count / total_rows) * 100
            
            if null_rate > 5:
                issues.append(f"High null rate in {col_name}: {null_rate:.2f}%")
    
    # Check 3: Duplicate event IDs
    distinct_ids = df.select(countDistinct("event_id_cnty")).collect()[0][0]
    if distinct_ids < total_rows:
        duplicates = total_rows - distinct_ids
        issues.append(f"Found {duplicates} duplicate event_id_cnty values")
    
    # Check 4: Future dates
    future_dates = df.filter(to_date(col("event_date")) > current_date()).count()
    if future_dates > 0:
        issues.append(f"Found {future_dates} events with future dates")
    
    return {"total_rows": total_rows, "issues": issues}

print("✓ Validation functions loaded")

In [0]:
# -----------------------------
# 2. Check for existing data (incremental load)
# -----------------------------

# Check if the bronze table already exists
try:
    max_timestamp_df = spark.sql("""
        SELECT MAX(timestamp) as max_ts
        FROM info_env_jordan.bronze.acled_jordan_events
    """)
    
    max_timestamp = max_timestamp_df.collect()[0]['max_ts']
    
    if max_timestamp:
        print(f"Existing data found. Latest timestamp: {max_timestamp}")
        print(f"Latest date: {datetime.fromtimestamp(max_timestamp)}")
        # Add small buffer to avoid missing records (subtract 1 day = 86400 seconds)
        last_sync_timestamp = max_timestamp - 86400
    else:
        print("Table exists but is empty. Performing full load.")
        last_sync_timestamp = None
        
except Exception as e:
    print(f"Table does not exist yet. Performing full load: {e}")
    last_sync_timestamp = None

print(f"Will pull records with timestamp > {last_sync_timestamp if last_sync_timestamp else 'all'}")

In [0]:
# -----------------------------
# 3. Pull ACLED data for Jordan (incremental)
# -----------------------------

api_url = "https://acleddata.com/api/acled/read"

# loop through last 5 years of data 
current_year = datetime.now().year
years = list(range(current_year - 4, current_year + 1))

all_records = []

for year in years:
    page = 1
    limit = 5000

    while True:
        params = {
            "country": "Jordan",
            "year": year,
            "limit": limit,
            "page": page,
            "_format": "json"
        }
        
        # Add timestamp filter for incremental loading
        if last_sync_timestamp:
            params["timestamp"] = last_sync_timestamp

        response = session.get(api_url, headers=headers, params=params)
        response.raise_for_status()

        data = response.json()
        records = data.get("data", [])

        if not records:
            break

        all_records.extend(records)

        print(f"Year {year}, page {page}: pulled {len(records)} records")

        if len(records) < limit:
            break

        page += 1

print(f"Total new records pulled: {len(all_records)}")

In [0]:
# -----------------------------
# 3. Convert to Spark DataFrame with Validation
# -----------------------------

if not all_records:
    print("No new records to process. Table is already up to date.")
    dbutils.notebook.exit("No new records")

pdf = pd.DataFrame(all_records)

display(pdf.head())

spark_df = spark.createDataFrame(pdf)

# Run validation checks
print(f"\n=== Data Quality Validation ===")

# Schema validation (checks for breaking changes)
validate_schema(spark_df, "info_env_jordan.bronze.acled_jordan_events")

# Data quality checks
quality_report = check_data_quality(spark_df)
print(f"Total rows: {quality_report['total_rows']}")

if quality_report['issues']:
    print("\n⚠️  Issues detected:")
    for issue in quality_report['issues']:
        print(f"  - {issue}")
    # Option: raise exception for critical issues if needed
    # if any("CRITICAL" in issue for issue in quality_report['issues']):
    #     raise ValueError("Critical data quality issues detected")
else:
    print("✓ All quality checks passed")

display(spark_df)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType

# Define expected schema for ACLED Jordan dataset (based on actual API response)
expected_schema = StructType([
    StructField("event_id_cnty", StringType(), True),
    StructField("event_date", StringType(), True),
    StructField("year", LongType(), True),
    StructField("time_precision", StringType(), True),
    StructField("disorder_type", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("sub_event_type", StringType(), True),
    StructField("actor1", StringType(), True),
    StructField("assoc_actor_1", StringType(), True),
    StructField("inter1", StringType(), True),
    StructField("actor2", StringType(), True),
    StructField("assoc_actor_2", StringType(), True),
    StructField("inter2", StringType(), True),
    StructField("interaction", StringType(), True),
    StructField("civilian_targeting", StringType(), True),
    StructField("iso", LongType(), True),
    StructField("region", StringType(), True),
    StructField("country", StringType(), True),
    StructField("admin1", StringType(), True),
    StructField("admin2", StringType(), True),
    StructField("admin3", StringType(), True),
    StructField("location", StringType(), True),
    StructField("latitude", StringType(), True),
    StructField("longitude", StringType(), True),
    StructField("geo_precision", LongType(), True),
    StructField("source", StringType(), True),
    StructField("source_scale", StringType(), True),
    StructField("notes", StringType(), True),
    StructField("fatalities", LongType(), True),
    StructField("tags", StringType(), True),
    StructField("timestamp", LongType(), True)
])

expected_columns = [field.name for field in expected_schema.fields]

# Check for missing or extra columns
actual_columns = pdf.columns.tolist()
missing_columns = [col for col in expected_columns if col not in actual_columns]
extra_columns = [col for col in actual_columns if col not in expected_columns]

if missing_columns or extra_columns:
    raise ValueError(f"Schema mismatch detected.\nMissing columns: {missing_columns}\nExtra columns: {extra_columns}")

print("Schema validation passed. Columns match expected schema.")

In [0]:
# -----------------------------
# 4. Write to bronze Delta table (merge for incremental)
# -----------------------------

from delta.tables import DeltaTable

spark.sql("CREATE SCHEMA IF NOT EXISTS info_env_jordan.bronze")

table_name = "info_env_jordan.bronze.acled_jordan_events"

# Check if table exists
table_exists = spark.catalog.tableExists(table_name)

if not table_exists:
    # First load: create table with partitioning
    print("Creating new table with year partitioning...")
    spark_df.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("year") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)
    print(f"Created {table_name} with {spark_df.count()} records")
else:
    # Incremental load: merge new/updated records
    print("Merging new records into existing table...")
    
    delta_table = DeltaTable.forName(spark, table_name)
    
    # Create temp view for merge
    spark_df.createOrReplaceTempView("new_acled_data")
    
    # Merge based on event_id_cnty (unique identifier)
    delta_table.alias("target").merge(
        spark_df.alias("source"),
        "target.event_id_cnty = source.event_id_cnty"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    
    print(f"Merged {spark_df.count()} records into {table_name}")

print(f"\nTable {table_name} update complete.")

In [0]:
# -----------------------------
# 6. Log Changes & Refresh Monitoring (Optional)
# -----------------------------

# If you've set up Lakehouse Monitoring, refresh metrics
try:
    from databricks import lakehouse_monitoring as lm
    
    # Refresh monitoring metrics after ingestion
    lm.run_refresh(table_name="info_env_jordan.bronze.acled_jordan_events")
    print("✓ Monitoring metrics refreshed")
except Exception as e:
    print(f"Monitoring not configured yet (optional): {e}")

# Log summary
final_stats = spark.sql("""
    SELECT 
        COUNT(*) as total_records,
        MAX(event_date) as latest_event,
        MAX(timestamp) as latest_timestamp
    FROM info_env_jordan.bronze.acled_jordan_events
""")

print("\n=== Final Table Stats ===")
display(final_stats)

In [0]:
# -----------------------------
# 5. Verify incremental load results
# -----------------------------

# Get summary statistics
summary = spark.sql("""
    SELECT 
        COUNT(*) as total_records,
        MIN(year) as earliest_year,
        MAX(year) as latest_year,
        MIN(event_date) as earliest_event,
        MAX(event_date) as latest_event,
        MAX(timestamp) as latest_timestamp
    FROM info_env_jordan.bronze.acled_jordan_events
""")

print("\n=== Table Summary ===")
display(summary)

# Show recent events by year
recent_by_year = spark.sql("""
    SELECT 
        year,
        COUNT(*) as event_count
    FROM info_env_jordan.bronze.acled_jordan_events
    GROUP BY year
    ORDER BY year DESC
""")

print("\n=== Events by Year ===")
display(recent_by_year)